# 09 - External ECG/CXR ablation design and configuration

This experiment separates two questions:

1. **Encounter bridge:** which Symile objective best transfers matched ECG-CXR evidence?
2. **Semantic re-alignment:** can independent ECG-diagnosis and CXR-report pairs be mapped into the frozen bridge?

Same-admission Symile pairs are hard positives for provenance, not proof of diagnostic equivalence. External ECG and CXR collections are not assumed to contain paired instances. Exact cross-sensor retrieval is evaluated only when shared `hadm_id` or `subject_id` values are available; otherwise cross-sensor evaluation uses explicitly labeled semantic compatibility and is reported separately.

### Bridge ablations

| Variant | Continuous alignment | Shared VQ | Specific VQ | Code matching |
|---|---:|---:|---:|---:|
| `clip_continuous` | yes | no | no | no |
| `shared_vq` | yes | yes | no | yes |
| `shared_specific_vq` | yes | yes | yes | yes |
| `shared_specific_no_code_match` | yes | yes | yes | no |
| `shuffled_pairs_control` | yes | yes | yes | yes, but ECG-CXR training pairs are shuffled |

The current project is a CodeBind-inspired implementation: it tests shared/specific compositional VQ, but does not claim reproduction of every component in the original paper (for example, its reconstruction decoder and EMA codebook maintenance).

### External semantic ablations

- `frozen_zero_shot`: no external adaptation; sensor codes only.
- `shared_raw_text`: one text adapter shared by ECG machine diagnoses and CXR reports.
- `separate_raw_text`: independent ECG-text and CXR-text adapters.
- `shared_concepts`: one shared adapter trained on normalized concept strings; rule-derived by default and replaceable by audited LLM output.

Use at least three seeds for final comparisons. Model selection uses Symile validation data or the external validation split, never external test results.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
config_path = ARTIFACT_ROOT / "config.json"
if not config_path.is_file():
    raise FileNotFoundError(f"Run notebook 00 or 01 first; missing {config_path}")
config = json.loads(config_path.read_text())

additions = {
    "external_ecg_root": os.environ.get(
        "MIMIC_ECG_ROOT", "/path/to/user/ecg_analysis/ecg_files/mimic-iv-ecg/1.0"),
    "external_cxr_root": os.environ.get(
        "MIMIC_CXR_TRAIN_ROOT",
        os.environ.get("MIMIC_CXR_JSON_ROOT", "/path/to/user/openi/multi_kg/train")),
    "external_cxr_test_root": os.environ.get(
        "MIMIC_CXR_TEST_ROOT", "/path/to/user/openi/multi_kg/test"),
    "text_model_id": os.environ.get(
        "CLINICAL_TEXT_MODEL_ID",
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"),
    "ablation_variants": [
        "clip_continuous", "shared_vq", "shared_specific_vq",
        "shared_specific_no_code_match", "shuffled_pairs_control",
    ],
    "ablation_seeds": [43, 44, 45],
    "external_splits": {"train": 80, "val": 10, "test": 10},
}
config.update(additions)
for name in ["ablation", "external/ecg", "external/cxr", "external/text", "external/adapters", "reports/ablation"]:
    (ARTIFACT_ROOT / name).mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config, indent=2))
print(json.dumps(additions, indent=2))


In [ ]:
required = ["numpy", "pandas", "PIL", "torch", "tqdm", "transformers", "wfdb", "scipy", "open_clip"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print("Missing packages:", missing)
    print("Install in the Biowulf environment, for example: pip install wfdb scipy open_clip_torch")
assert Path(config["external_ecg_root"]).exists(), config["external_ecg_root"]
assert Path(config["external_cxr_root"]).exists(), config["external_cxr_root"]
assert Path(config["external_cxr_test_root"]).exists(), config["external_cxr_test_root"]
print("Configuration ready. Missing optional packages:", missing)
